# Evaluación Sumativa 2 — Validación por remuestreo y simulación
## Dataset: *E-Commerce Shipping Data* (entregas atrasadas)

**MCDI501 - Estadística Computacional para la Toma de Decisiones**
Magíster en Ciencia de Datos e Inteligencia Artificial, Universidad Andrés Bello
Equipo: José Ignacio Rodríguez, Yerko Gallardo Miranda y Sebastián Rojas Araya

Este cuaderno valida **todos** los resultados de la Sumativa 1 (S1) de cada familia mediante bootstrap, permutación, simulación Monte Carlo, jackknife y análisis de sensibilidad, y prepara insumos verificables para la Sumativa 3 (S3). Sigue el orden de la planificación del equipo (versión 5):

0. Configuración, datos y semilla
1. Punto de partida, alcance y cobertura
2. Bootstrap y pruebas de permutación
3. Correlaciones, asociaciones y simulación Monte Carlo
4. Robustez, sensibilidad e insumos para S3
5. Comprobaciones de aceptación

El cuaderno desarrolla la cobertura completa. El informe presenta en su cuerpo principal (máximo siete páginas) los hallazgos más relevantes, y en los anexos A–F el desarrollo completo.

## 0. Configuración, datos y semilla

Los datos se leen directamente desde S1, sin copiarlos: `sumativa_1/data/raw/ecommerce_shipping.csv`. Antes de usarlos se verifica su huella SHA-256 contra `sumativa_1/docs/registro_fuente.json`, que acepta la forma original (CRLF) y la normalizada (LF) del archivo.

La base de trabajo `df` usa los mismos nombres de variables y la misma codificación que S1: `entrega_atrasada = 1` para atraso y `0` para entrega puntual; las diferencias entre grupos se calculan como **atrasadas menos puntuales**. No se eliminan filas ni se modifica el CSV.

In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd

# Raíz de la Sumativa 2: funciona al ejecutar desde el repositorio, sumativa_2/ o sus notebooks/
ubicacion = Path.cwd().resolve()
candidatas = [p for a in [ubicacion, *ubicacion.parents] for p in [a, a / 'sumativa_2']]
RAIZ = next((p for p in candidatas if (p / 'notebooks/sumativa_2_grupo1.ipynb').is_file()
             and (p / 'docs/registro_semilla.json').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Ejecute desde el repositorio, sumativa_2/ o sus notebooks/.')
RAIZ_S1 = RAIZ.parent / 'sumativa_1'
DATOS = RAIZ_S1 / 'data/raw/ecommerce_shipping.csv'

# Integridad: el CSV debe ser el registrado en S1 (forma original CRLF o normalizada LF)
registro_fuente = json.loads((RAIZ_S1 / 'docs/registro_fuente.json').read_text(encoding='utf-8'))
HUELLA_DATOS = hashlib.sha256(DATOS.read_bytes()).hexdigest()
assert HUELLA_DATOS in {registro_fuente['file']['sha256'], registro_fuente['file']['sha256_lf_normalized']}, \
    'El CSV no coincide con el registrado en S1.'

raw = pd.read_csv(DATOS, encoding='utf-8-sig')

NOMBRES = {
    'ID': 'id_envio', 'Warehouse_block': 'bloque_bodega', 'Mode_of_Shipment': 'modalidad_envio',
    'Customer_care_calls': 'llamadas_atencion', 'Customer_rating': 'calificacion_cliente',
    'Cost_of_the_Product': 'costo_producto', 'Prior_purchases': 'compras_previas',
    'Product_importance': 'importancia_producto', 'Gender': 'genero',
    'Discount_offered': 'descuento_ofrecido', 'Weight_in_gms': 'peso_gramos',
}
TARGET = 'entrega_atrasada'

df = raw.rename(columns=NOMBRES)
df[TARGET] = raw['Reached.on.Time_Y.N']
df['importancia_producto'] = pd.Categorical(df['importancia_producto'],
                                            categories=['low', 'medium', 'high'], ordered=True)

n = len(df)
n_atr, n_punt = int(df[TARGET].sum()), int((df[TARGET] == 0).sum())
assert (n, n_atr, n_punt) == (10_999, 6_563, 4_436), 'Los conteos no coinciden con S1.'

print(f'Datos: {DATOS.relative_to(RAIZ.parent).as_posix()} (SHA-256 {HUELLA_DATOS[:12]}…)')
print(f'Dimensiones: {n:,} filas x {df.shape[1]} columnas | atrasadas: {n_atr:,} | puntuales: {n_punt:,}'.replace(',', '.'))
df.head()

Datos: sumativa_1/data/raw/ecommerce_shipping.csv (SHA-256 dcdc3fd8f050…)
Dimensiones: 10.999 filas x 13 columnas | atrasadas: 6.563 | puntuales: 4.436


,id_envio,bloque_bodega,modalidad_envio,llamadas_atencion,calificacion_cliente,costo_producto,compras_previas,importancia_producto,genero,descuento_ofrecido,peso_gramos,Reached.on.Time_Y.N,entrega_atrasada
0,1,D,Flight,4,2,177,3,low,F,44,1233,1,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1,1


### Semilla

Toda la aleatoriedad del cuaderno usa la **semilla única** registrada según `docs/protocolo_semilla.md`, obtenida del *beacon* de aleatoriedad de la Universidad de Chile con una regla publicada antes de la emisión del pulso. Se lee desde `docs/registro_semilla.json`: no se escribe a mano, no se reemplaza por otra (por ejemplo, 42) ni se genera una nueva.

Aquí solo se comprueba que la semilla corresponde al `outputValue` del pulso registrado. La verificación completa contra el *beacon* se ejecuta con `uv run python sumativa_2/scripts/semilla_beacon.py verificar`.

In [2]:
registro_semilla = json.loads((RAIZ / 'docs/registro_semilla.json').read_text(encoding='utf-8'))
SEMILLA = registro_semilla['semilla']

# La semilla debe derivarse del pulso registrado: int(outputValue, 16) mod 2**32
assert SEMILLA == int(registro_semilla['pulso']['outputValue'], 16) % 2**32, \
    'La semilla no corresponde al pulso registrado.'

print(f'SEMILLA = {SEMILLA}')
print(f"Pulso: {registro_semilla['pulso']['uri']} ({registro_semilla['pulso']['timeStamp']})")

SEMILLA = 1505514103
Pulso: https://random.uchile.cl/beacon/2.1-beta/chain/2/pulse/1053589 (2026-10-14T15:00:00.000Z)


### Flujos aleatorios

Cada módulo recibe su propio generador, derivado de la semilla con `np.random.SeedSequence(SEMILLA).spawn(k)`. Así, los resultados de un módulo no dependen de las llamadas aleatorias de otro, y cada sección puede ejecutarse o revisarse por separado.

El orden de los flujos se fija y se documenta **antes** de las ejecuciones finales:

| Flujo | Módulo | Sección |
|---|---|---|
| *por asignar* | Bootstrap de todos los parámetros y efectos | 2.1 |
| *por asignar* | Permutaciones de las siete pruebas | 2.2 |
| *por asignar* | Bootstrap de las 20 correlaciones y 15 asociaciones | 3.1 |
| *por asignar* | Monte Carlo de lotes de envíos | 3.2 |
| *por asignar* | Monte Carlo empírico de todos los estimandos (Anexo D) | 3.2 |
| *por asignar* | Bootstrap y permutaciones de los escenarios de sensibilidad | 4.2 |
| *por asignar* | Sensibilidad de Monte Carlo | 4.3 |

Junto con el mapa de flujos se registran la semilla, n, B, N, α, las versiones de los paquetes, los tiempos y las advertencias.

## 1. Punto de partida, alcance y cobertura

**Objetivo.** Validar resultados propios de S1 sobre atrasos en envíos de comercio electrónico, cuantificar su incertidumbre y su sensibilidad, y preparar insumos verificables para S3. Los valores de referencia son resultados de S1, no de S2.

**Referencias de S1.** Se leen desde `sumativa_1/outputs/tablas/` y se registran en `docs/manifiesto_s1.json`, con el commit, las rutas y las huellas de cada archivo. Los cuatro parámetros inferenciales principales son la proporción de atraso (IC de Wilson) y las medias de peso, costo y llamadas (IC *t* de Student), en `estimaciones.csv`.

### Regla de cobertura completa

Cada técnica se aplica a **todos los resultados de S1 de la familia a la que corresponde**. Ningún resultado se excluye del cálculo porque no se destaque en el cuerpo del informe. El inventario comprende:

- las cuatro estimaciones inferenciales principales;
- todos los IC y efectos de los contrastes;
- las estadísticas descriptivas y las proporciones por grupo;
- las siete pruebas;
- 20 coeficientes de correlación distintos y 15 asociaciones V de Cramér.

Los resultados repetidos entre tablas se deduplican, conservando el vínculo con cada fuente, y las cantidades se verifican contra la línea base. Una técnica no se aplica mecánicamente donde no es válida: se registra el método, el resultado o el motivo de no aplicabilidad. Por ejemplo, los IC *bootstrap* de valores extremos o de variables constantes requieren advertencia, y la V de Cramér es una asociación no negativa, no una correlación con signo.

### Desarrollo completo en anexos

| Anexo | Contenido |
|---|---|
| A | Todos los parámetros y sus IC |
| B | Las siete pruebas, sus distribuciones nulas y Holm |
| C | 20 correlaciones, 15 asociaciones y atlas de distribuciones |
| D | Todos los experimentos de Monte Carlo |
| E | Sensibilidad, jackknife e IDs influyentes |
| F | Matriz completa de resultados para S3 |

La selección de lo que va al cuerpo se justifica en `matriz_editorial.csv` (ID, pregunta, aporte, ubicación y criterio de rúbrica), sin elegir por valor *p*, signo favorable o facilidad de representación.

**Decisiones de alcance.** No se introduce otro conjunto de datos ni se ajusta un modelo predictivo nuevo.

**Limitaciones.** La inferencia sigue condicionada a la independencia y la representatividad de los registros, que el origen del conjunto de datos no permite asegurar. El remuestreo no corrige sesgos de selección ni demuestra causalidad.

*Pendiente (Y01): inventario completo de S1 (`inventario_resultados_s1.csv`), huellas y matriz de cobertura en `docs/manifiesto_s1.json`.*

## 2. Bootstrap y pruebas de permutación

### 2.1 Bootstrap de todos los parámetros y efectos

**Responsable:** Yerko · **Revisor:** Sebastián · **Desarrollo completo:** Anexo A

**Método.** Se procesa el inventario completo con **10.000 remuestras por resultado** e IC percentil y BCa al 95 %, conservando el estimador, la unidad, la población y los grupos de S1:

- **Parámetros globales:** se remuestrean filas completas, de tamaño 10.999.
- **Diferencias y efectos entre grupos:** se remuestrea dentro de cada grupo, manteniendo sus tamaños.
- Las réplicas se comparten dentro de una misma familia cuando corresponde.

| Familia de S1 | Cobertura y comparación |
|---|---|
| `estimaciones.csv` / `intervalos_confianza.csv` | Los cuatro parámetros (atraso, peso, costo y llamadas), comparados con el IC de Wilson o *t* original |
| Contrastes y `comparaciones_proporciones.csv` | Diferencias de peso, descuento y costo; diferencias de proporciones de E2 y E3; efectos $d_{av}$, riesgo relativo y biserial de rangos |
| Descriptivas, frecuencias y tasas por grupo | Todas las medias, medianas, dispersiones, cuantiles, proporciones y tasas publicadas, identificando cuáles no tenían IC clásico |
| Extremos, constantes y fronteras | Mínimo y máximo como diagnóstico descriptivo, sin atribuir cobertura poblacional al *bootstrap* ordinario de extremos; se registran las degeneraciones y los BCa no estimables |

**Inventario.** `inventario_resultados_s1.csv` registra ID, fuente, variable o grupo, estimación, IC original y método de S2. Si no hay IC clásico, se marca «no disponible en S1». Los conteos y proporciones redundantes se vinculan mediante n; un n fijo no se remuestrea.

**BCa.** Trata empates y estima la aceleración por *jackknife*; se valida contra una implementación de referencia y se procesa por lotes. No se sustituye *t* por *z* ni se supone robustez automática; se conservan las unidades y los estimandos.

**Evidencias:** `outputs/tablas/bootstrap_intervalos.csv` (una fila por estimando: n/B, estimación de S1, sesgo, EE, $z_0$, aceleración, IC percentil y BCa, y diferencias de extremos y ancho), distribuciones de todos los estimandos y un atlas en el Anexo A.

**Para el cuerpo del informe** *(pendiente)*: al menos tres parámetros relevantes, cada uno con IC clásico, percentil y BCa, su distribución e interpretación crítica. Propuesta: proporción de atraso (riesgo operativo), peso medio (perfil logístico) y costo medio (perfil económico); las llamadas solo si aportan una conclusión necesaria sobre atención.

### 2.2 Permutación de las siete pruebas

**Responsable:** Sebastián · **Revisor:** Yerko · **Desarrollo completo:** Anexo B

**Método.** Cada contraste conserva su $H_0/H_1$, sus grupos, su dirección y su estadístico de S1. Se ejecutan **10.000 permutaciones por prueba**, recalculando siempre el estadístico:

| Prueba de S1 | Estadístico y esquema de permutación |
|---|---|
| T01: modalidad–atraso | $\chi^2$ de Pearson; se permuta el atraso entre modalidades, con márgenes fijos |
| T02: peso por atraso | *t* de Welch studentizada; se permutan las etiquetas de atraso |
| T03: descuento por atraso | *t* de Welch studentizada; mismo esquema |
| E1: peso por atraso | *U* de Mann–Whitney con empates; se permutan las etiquetas y se centra en $n_1 n_0/2$ |
| E2: atraso, importancia alta frente al resto | *z* de dos proporciones; se permuta el atraso entre grupos fijos |
| E3: atraso, descuento > 10 frente a ≤ 10 | *z* de dos proporciones; mismo esquema, advirtiendo la frontera $p_1 = 1$ |
| E4: costo por atraso | *t* de Welch studentizada; se permutan las etiquetas de atraso |

**Valor p.** Para *t* y *z* bilaterales se cuentan las excedencias de $|\text{estadístico}|$; para *U* bilateral, la distancia al centro $n_1 n_0/2$; para $\chi^2$, la cola superior:

$$p_{perm} = \frac{1 + \#\{\text{excedencias}\}}{B + 1}$$

Se guardan la fórmula, los empates y la resolución $1/10.001$. Los *p* registrados como 0 en S1 son subdesbordamiento numérico, no probabilidad exactamente nula.

**Multiplicidad.** Se aplica **Holm a la familia completa de los siete *p* de permutación** y se comparan las decisiones con las de la familia de Holm original, separando las comparaciones crudas de las ajustadas. Con B = 10.000 se reconoce una resolución limitada y una posible incertidumbre de decisión cerca del umbral; si hace falta, se amplía B con una regla documentada.

**Supuestos.** La exactitud requiere intercambiabilidad; la *t* de Welch studentizada tiene justificación asintótica bajo heterocedasticidad; Mann–Whitney contrasta distribuciones o rangos, no igualdad de medias.

**Evidencias:** `outputs/tablas/permutaciones_todas.csv` y las siete distribuciones nulas (Anexo B).

**Para el cuerpo del informe** *(pendiente)*: al menos una prueba relevante, T02 (peso por atraso), con $H_0/H_1$, efecto, estadístico observado, distribución nula, *p* original y de permutación, y discusión de supuestos. Se agrega otra solo si aporta una respuesta distinta y útil al problema; la comparación completa y Holm quedan en el Anexo B, conservando para las pruebas reportadas la referencia al ajuste familiar.

## 3. Correlaciones, asociaciones y simulación Monte Carlo

### 3.1 Todas las correlaciones y asociaciones

**Responsable:** Sebastián · **Revisor:** José · **Desarrollo completo:** Anexo C

**Universo completo:**

| Fuente de S1 | Coeficientes |
|---|---|
| `correlacion_spearman.csv` | 10 pares únicos entre cinco variables cuantitativas |
| `correlacion_dependiente_cuantitativas.csv` | 5 punto-biseriales y 5 Spearman con el atraso |
| `asociacion_cramer.csv` | 10 pares únicos entre cinco variables categóricas |
| `relacion_dependiente_cualitativas.csv` | 5 asociaciones V de Cramér con el atraso |

**Resultado esperado:** 20 correlaciones y 15 asociaciones, con estimación de S1, n/B, EE, IC, ancho, diagnóstico y sensibilidad. Las asociaciones repetidas en T01 se validan una vez y se referencian desde ambos apartados.

**Método.** Se remuestrean filas emparejadas **10.000 veces por coeficiente**, con IC percentil al 95 %. Se conserva el tipo de coeficiente y se recalculan los rangos de Spearman en cada remuestra. Se excluyen las diagonales y los pares simétricos duplicados, y nunca se remuestrean las columnas por separado.

**Interpretación** *(pendiente)*.

- **Correlaciones:** inclusión del cero, signo y precisión; distinguir un efecto débil de uno inestable.
- **V de Cramér**, acotada en [0, 1]: no se traslada la lógica del signo. Se informan la magnitud, el ancho y el sesgo del *bootstrap* cerca de cero; la independencia no se resuelve solo con un IC percentil positivo.

**Evidencias:** `outputs/tablas/correlaciones_bootstrap.csv`, `outputs/tablas/asociaciones_bootstrap.csv`, atlas de distribuciones y todos los IC (Anexo C).

**Para el cuerpo del informe** *(pendiente)*: entre tres y cinco correlaciones de mayor relevancia para los atrasos, con distribución, IC 95 %, estabilidad y magnitud. Propuesta inicial: atraso–descuento y atraso–peso (punto-biseriales) y descuento–peso (Spearman). Dos coeficientes del mismo par no cuentan como dos hallazgos, y no se agregan 35 pruebas nuevas a la familia de Holm original.

### 3.2 Monte Carlo y extensión a todos los parámetros

**Responsable:** José · **Revisor:** Yerko · **Desarrollo completo:** Anexo D

**Pregunta.** ¿Qué variabilidad del porcentaje de atrasos cabe esperar en lotes comparables al conjunto de datos, y cómo cambia con el tamaño del lote?

**Modelo.** $K \sim \text{Binomial}(N, p)$ con $p = 6.563/10.999$, la proporción de S1, y tasa $K/N$. Tamaños de lote $N$ = 100, 500, 1.000 y 5.000 (una decisión de escenario, no un parámetro estimado), con **10.000 lotes por tamaño**.

**Resultados por tamaño:** media, desviación estándar, cuantiles 2,5, 97,5 y 99, y $P(\text{tasa} > 0{,}60)$. El 60 % es un umbral ilustrativo, no un objetivo empresarial acreditado.

**Controles internos:** media teórica $p$, varianza $p(1-p)/N$ y probabilidad exacta binomial. La coincidencia demuestra la coherencia del simulador, no una validación externa del modelo.

**Convergencia.** Media y probabilidad acumuladas para B = 100, 500, 1.000, 5.000 y 10.000, con $\text{MCSE}_{media} = s(\text{tasa})/\sqrt{B}$ y $\text{MCSE}_{prob} = \sqrt{\hat q(1-\hat q)/B}$. Tolerancias fijadas antes de ejecutar: $\text{MCSE}_{media} \le 0{,}001$ y $\text{MCSE}_{prob} \le 0{,}005$, junto con la diferencia frente al valor analítico. Si no se cumplen, se amplía hasta 50.000 lotes por escenario y se documenta. Si un evento no ocurre en ninguna simulación, se reporta su límite binomial en vez de afirmar incertidumbre nula.

**Interpretación** *(pendiente)*. Los cuantiles forman un **intervalo de resultados simulados condicionado a $p$**, no un IC del parámetro $p$; la incertidumbre de estimación se examina aparte, variando $p$ dentro del IC de S1 (sección 4.3).

**Extensión a todos los estimandos (Anexo D).** Para cada estimando del inventario se evalúa su comportamiento por Monte Carlo usando remuestras conjuntas de la distribución empírica de S1, con B ≥ 10.000. Las réplicas del *bootstrap* pueden reutilizarse como experimento de Monte Carlo explícitamente identificado, sin contarlas como evidencia independiente. Se reportan media, dispersión, cuantiles, sesgo respecto del valor empírico y MCSE cuando esté definido. Se conservan la dependencia, la estructura de grupos y los dominios: la simulación operativa principal sigue siendo binomial, no se inventan distribuciones paramétricas ni se generan variables independientes para «usar todos los parámetros». La referencia poblacional de esta extensión es la distribución empírica, no una población externa conocida.

**Evidencias:** `outputs/tablas/montecarlo_escenarios.csv`, `outputs/tablas/montecarlo_convergencia.csv` y figuras de distribución y convergencia.

**Para el cuerpo del informe** *(pendiente)*: el lote N = 1.000, su riesgo de atraso, la convergencia y la sensibilidad relevante. Se comparan tamaños solo si cambia la interpretación o la recomendación; el resto de tamaños, diagnósticos y experimentos va en el Anexo D.

## 4. Robustez, sensibilidad e insumos para S3

### 4.1 Observaciones influyentes (jackknife)

**Responsable:** Yerko · **Revisor:** José · **Desarrollo completo:** Anexo E

**Método.** Se conserva el análisis completo como referencia. Se aplica *jackknife* a todos los estimandos y coeficientes del inventario que admitan eliminar una fila: error estándar *jackknife* y cambio de cada estimación al retirar cada envío.

- **Medias:** se usa la identidad $\bar x_{(-i)} = (\text{suma} - x_i)/(n-1)$, que evita un cálculo cuadrático.
- **Efectos entre grupos:** se conserva la pertenencia al grupo y se actualizan los tamaños.
- **Pruebas:** se evalúa el cambio del efecto o del estadístico y de la decisión, sin interpretar el *jackknife* del valor *p* como un IC de un parámetro.
- **Casos indefinidos:** se registra la causa.

**Registro:** los diez registros más influyentes por estadístico, con `id_envio`, variable, valor original y cambio absoluto y relativo. Los empates se resuelven por ID y se informa que existen.

**Criterio.** Un valor extremo estadísticamente válido no se elimina por defecto.

**Interpretación** *(pendiente)*.

### 4.2 Sensibilidad a extremos y estructura

**Responsables:** Yerko y Sebastián · **Revisores:** José y Yerko · **Desarrollo completo:** Anexo E

**Escenarios**, evaluados por separado:

1. Datos completos (referencia).
2. Winsorización 1/99 de peso y costo.
3. Exclusión exploratoria de observaciones fuera de 1,5·IQR.
4. Análisis restringido a descuento ≤ 10.

Para cada escenario se registran la regla, los IDs afectados, el n retenido y los cambios en todos los estimandos, los IC, las siete pruebas, las 20 correlaciones y las 15 asociaciones. El CSV no se modifica, y los análisis restringidos no se tratan como estimaciones de la misma población.

**Recálculo por escenario.** Se repiten los IC *bootstrap* y las siete permutaciones en cada escenario válido, con B = 10.000, y se recalcula Holm para la familia completa del escenario. Si un escenario suprime un grupo (por ejemplo, E3 al restringir a descuento ≤ 10), se marca como «no evaluable por diseño», sin fabricar un contraste ni un resultado.

**Diagnósticos.** La comparación de la media con la mediana o la media recortada indica un cambio de estimando, no valida directamente el IC de la media.

**Segmento de descuento > 10.** En S1, los 2.647 envíos con descuento mayor que 10 están atrasados, frente a 3.916 de 8.352 con descuento ≤ 10 ($p = 0{,}468870$). Un *bootstrap* del primer segmento produce solo unos: IC [1, 1] y BCa posiblemente indefinido. No se interpreta como certeza poblacional; se conserva como advertencia el IC de Wilson de S1, [0,998551; 1]. El umbral no se cambia buscando el resultado más favorable.

**Para el cuerpo del informe** *(pendiente)*: qué conclusiones centrales se sostienen o cambian, con la magnitud y la evidencia de influencia, incluida toda discrepancia material que afecte la interpretación, aunque contradiga la conclusión inicial. Todos los cambios se publican en el Anexo E; no se trasladan al cuerpo todos los escenarios.

### 4.3 Sensibilidad de Monte Carlo

**Responsable:** José · **Revisor:** Sebastián · **Desarrollo completo:** Anexo D

**Sensibilidad a $p$.** Se varía $p$ en los extremos y el centro del IC de Wilson de S1, con $N = 1.000$ y $B \ge 10.000$ por escenario. Se miden los cambios en la media, el percentil 99 y $P(\text{tasa} > 0{,}60)$, y se presentan en un gráfico de sensibilidad. Esta actividad evidencia el indicador ID2.3.

**Otras entradas.** Para cada entrada adicional con IC en S1, se varían el centro y los extremos y se registra el efecto sobre su salida pertinente; si no hay IC en S1, se usan escenarios empíricos identificados como extensión. No se agregan parámetros ajustados fuera de S1.

**Contraste de estructura.** Se simula una composición fija de los dos segmentos de descuento, a partir de sus tamaños y tasas en S1:

- $N_{alto} = \text{redondeo}(N \cdot 2.647/10.999)$
- $K_{alto} \sim \text{Binomial}(N_{alto}, 1)$
- $K_{bajo} \sim \text{Binomial}(N - N_{alto},\ 3.916/8.352)$

Su dispersión se contrasta con la del modelo global. Se explicita que $p_{alto} = 1$ es una frontera muestral y una limitación.

**Interpretación** *(pendiente)*.

### 4.4 Reporte de resultados validados

**Responsable:** José · **Revisor:** Yerko · **Desarrollo completo:** Anexo F

**Archivos:** `outputs/resultados_validados.csv` (entrada a S3) y su síntesis legible, `docs/resultados_validados.md`.

**Campos por resultado:** ID, fuente y valor en S1, método en S2, n/B, estimación, IC o *p* en S2, diagnóstico de sensibilidad, estado, limitación, recomendación para S3, evidencia, responsable, revisor, commit y semilla.

**Estados permitidos:** *pendiente*, *respaldado bajo supuestos*, *sensible*, *no concluyente* y *no evaluable con justificación*. Se asignan solo después de ejecutar y revisar.

**Cobertura.** Se comprueba una cobertura del 100 % del inventario: cada resultado tiene una validación o una excepción metodológica explícita. El Anexo F contiene la matriz completa; el cuerpo del informe incluye únicamente los resultados y recomendaciones para S3 que derivan de la discusión central, con evidencia e incertidumbre.

**Orientación para S3:** variables a conservar, registros a inspeccionar, tratamientos a comparar y validación externa necesaria. No se declara causalidad, independencia ni transportabilidad a partir de la concordancia entre métodos.

## 5. Comprobaciones de aceptación

Antes del congelamiento final se verifica:

- [ ] Huellas y referencias de S1; los valores de referencia se reproducen con una tolerancia explícita de punto flotante.
- [ ] B ≥ 10.000 en cada técnica; dominios e IC válidos (coeficientes en [−1, 1] y V de Cramér en [0, 1]).
- [ ] Colas de *t*, *z*, *U* y $\chi^2$; Holm; emparejamiento del *bootstrap* y advertencias de BCa.
- [ ] Monte Carlo: MCSE, convergencia y controles binomiales.
- [ ] Kernel reiniciado y todas las celdas ejecutadas en orden, repetido en un segundo equipo; cifras cotejadas con las tablas exportadas.
- [ ] Cobertura completa en los anexos y, en el cuerpo del informe, al menos tres parámetros, una prueba y entre tres y cinco correlaciones con evidencia y discusión, dentro de siete páginas.